# GPU Matrix Multiplication Benchmark (C + CUDA)

Runs the CPU baseline and the three GPU versions (naive, shared-memory tiled, cuBLAS), checks every GPU result against the CPU, and plots performance.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order.

In [ ]:
!nvidia-smi

## 1. Get the code
Run the upload cell and select `matmul_cpu.c`, `matmul_gpu.cu`, and `Makefile`.
Once the project is on GitHub, you can instead uncomment the `git clone` lines.

In [ ]:
import os
from google.colab import files

os.makedirs('src', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    dest = name if name == 'Makefile' else f'src/{name}'
    os.replace(name, dest)
print(sorted(os.listdir('src')))

# Alternative after you push to GitHub:
# !git clone https://github.com/Paneric-unc/gpu-matmul-benchmark.git
# %cd gpu-matmul-benchmark

## 2. Build

In [ ]:
!nvcc --version | tail -n 2
!make

## 3. Run
The CPU benchmark compares two loop orders. The GPU benchmark prints the GPU model and copy times, then a CSV of results.

In [ ]:
!./matmul_cpu 256 512 1024 | tee cpu_results.csv

In [ ]:
!./matmul_gpu 256 512 1024 2048 | tee gpu_results.csv

In [ ]:
# Larger sizes: GPU only (CPU check is skipped above 2048 because it takes too long)
!./matmul_gpu 4096 | tee gpu_results_large.csv

## 4. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cpu = pd.read_csv('cpu_results.csv')
gpu = pd.concat([pd.read_csv('gpu_results.csv'), pd.read_csv('gpu_results_large.csv')])
display(cpu)
display(gpu)

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, grp in pd.concat([cpu[cpu.version == 'cpu_ijk'], gpu]).groupby('version'):
    ax.plot(grp.n, grp.gflops, marker='o', label=name)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Matrix size n (n x n)')
ax.set_ylabel('GFLOPS (higher is better)')
ax.set_title('Matrix multiplication throughput')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('results.png', dpi=150)
plt.show()

# Headline numbers for the README and resume
big = gpu[gpu.n == 2048].set_index('version')
print(f"n=2048: tiled vs naive speedup = {big.loc['gpu_naive','ms'] / big.loc['gpu_tiled','ms']:.2f}x")
print(f"n=2048: tiled vs CPU speedup   = {big.loc['gpu_tiled','speedup_vs_cpu']:.0f}x")
print(f"n=2048: tiled reaches {100 * big.loc['gpu_tiled','gflops'] / big.loc['cublas','gflops']:.0f}% of cuBLAS")

In [ ]:
# Download results to add to your GitHub repo
for f in ['cpu_results.csv', 'gpu_results.csv', 'gpu_results_large.csv', 'results.png']:
    files.download(f)